## rag has 3 components
- retrieval (data -> vector database)
- augumented - setting  plus vector into prompt
- generation - output generation

In [ ]:
!pip install -q -U google-generativeai

In [ ]:
import google.generativeai as genai
from google.colab import userdata

genai.configure(api_key=userdata.get('GOOGLE_API_KEY'))

- google colab - client
- model - server

In [ ]:
list(genai.list_models())
# models available on google server

[Model(name='models/embedding-gecko-001',
       base_model_id='',
       version='001',
       display_name='Embedding Gecko',
       description='Obtain a distributed representation of a text.',
       input_token_limit=1024,
       output_token_limit=1,
       supported_generation_methods=['embedText', 'countTextTokens'],
       temperature=None,
       max_temperature=None,
       top_p=None,
       top_k=None),
 Model(name='models/gemini-2.5-pro-preview-03-25',
       base_model_id='',
       version='2.5-preview-03-25',
       display_name='Gemini 2.5 Pro Preview 03-25',
       description='Gemini 2.5 Pro Preview 03-25',
       input_token_limit=1048576,
       output_token_limit=65536,
       supported_generation_methods=['generateContent',
                                     'countTokens',
                                     'createCachedContent',
                                     'batchGenerateContent'],
       temperature=1.0,
       max_temperature=2.0,
       top_p=0.9

Tasks that rag can perform
- retrieval query
- retrieval documentation
- retrieval clasification
- retrieval clustering
- semantic similarity


In [ ]:
from typing import Dict

result : Dict = genai.embed_content(
    model = "models/text-embedding-004",
    content = "What is the meaning of life",
    task_type = "retrieval_document",
    title = "Embedding of a single string,"
)
result

{'embedding': [-0.022962498,
  0.05725789,
  -0.027628105,
  -0.006389069,
  -0.0350964,
  -0.0023710763,
  0.023161722,
  0.06488263,
  -0.0050077224,
  0.006761657,
  0.029883625,
  -0.02829802,
  0.0925115,
  -0.03161111,
  0.017438708,
  -0.113738686,
  0.004148322,
  0.0056179934,
  -0.09500403,
  -0.0124058975,
  0.016015321,
  -0.0032955993,
  0.05220307,
  -0.021110605,
  -0.0056991964,
  0.015071709,
  0.0014222017,
  -0.040490743,
  -0.012622519,
  -0.013441389,
  0.0702041,
  0.05568195,
  0.017926062,
  -0.029592672,
  0.052217267,
  0.032541398,
  -0.0067407046,
  0.064183526,
  0.024617577,
  -0.029804666,
  -0.07351093,
  0.021553325,
  -0.0421831,
  0.05780208,
  -0.02202611,
  -0.027217234,
  -0.018628336,
  -0.008581568,
  -0.025325883,
  0.05842717,
  0.022000082,
  0.028388556,
  -0.036370397,
  0.016413528,
  -0.02422371,
  -0.022113927,
  -0.013934325,
  -0.031934947,
  0.060141414,
  -0.047938444,
  -0.029832851,
  -0.026385073,
  -0.030777428,
  -0.014410568,
  

In [ ]:
len(result["embedding"])

768

Building vector store and retrieval using ChromaDB

In [ ]:
!pip install -Uq langchain-chroma

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 3.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.8/19.8 MB 88.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 73.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.3/103.3 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.3/17.3 MB 74.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.4/105.4 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 510.8/510.8 kB 36.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 128.4/128.4 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 101.9 MB/s eta 

In [ ]:
import getpass
import os

In [ ]:
from langchain_core.documents import Document # class
documents = [
    Document(
    page_content="A bunch of scientists bring back dinosaurs and mayhem breaks loose",
    metadata = {"source": "https://en.wikipedia.org/wiki/Jurassic_Park"},
    ),
    Document(
    page_content="Leo DiCaprio gets lost in a dream within a dream within a dream within a ...",
    metadata = {"source": "https://en.wikipedia.org/wiki/Oppenheimer_(film)"},
    ),
    Document(page_content="Leo DiCaprio gets lost in a dream within a dream within a dream within a ...",
    metadata = {"source": "https://en.wikipedia.org/wiki/Oppenheimer_(film)"},
    ),
]

In [ ]:
!pip install -Uq langchain-google-genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.7/50.7 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 17.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-generativeai 0.8.5 requires google-ai-generativelanguage==0.6.15, but you have google-ai-generativelanguage 0.7.0 which is incompatible.


In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(model="models/text-embedding-004",
                                          google_api_key=userdata.get('GOOGLE_API_KEY'))

In [ ]:
embeddings.embed_query("Who created this world?")

[-0.015027550980448723,
 0.017090151086449623,
 -0.05650967359542847,
 -0.026381397619843483,
 0.015025527216494083,
 0.019098665565252304,
 0.008009860292077065,
 0.0006964390631765127,
 -0.022438745945692062,
 0.052616994827985764,
 -0.022325647994875908,
 -0.04528185352683067,
 0.10773125290870667,
 0.0040437690913677216,
 -0.003893607296049595,
 -0.04018065705895424,
 0.004072260111570358,
 0.006672200281172991,
 -0.07493143528699875,
 0.02650458551943302,
 0.0472252182662487,
 -0.03813515231013298,
 0.02522432990372181,
 0.009686482138931751,
 0.0020060993265360594,
 0.027137205004692078,
 -0.0008476872462779284,
 -1.5059624274726957e-05,
 0.0009666663827374578,
 0.01287305261939764,
 0.03300386294722557,
 0.04181244596838951,
 0.018476124852895737,
 -0.04841843619942665,
 0.029258115217089653,
 -0.022181985899806023,
 -0.015090210363268852,
 0.016429463401436806,
 0.010490725748240948,
 -0.026336781680583954,
 -0.09767646342515945,
 0.04906834289431572,
 -0.015638770535588264,
 0

In [ ]:
len(embeddings.embed_query("Who created this world?"))

768

In [ ]:
from langchain_chroma import Chroma

vectorstore = Chroma.from_documents(documents, embeddings)
list(dir(vectorstore))

['_Chroma__ensure_collection',
 '_Chroma__query_collection',
 '_LANGCHAIN_DEFAULT_COLLECTION_NAME',
 '__abstractmethods__',
 '__annotations__',
 '__class__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__slots__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_abc_impl',
 '_asimilarity_search_with_relevance_scores',
 '_chroma_collection',
 '_client',
 '_collection',
 '_collection_configuration',
 '_collection_metadata',
 '_collection_name',
 '_cosine_relevance_score_fn',
 '_embedding_function',
 '_euclidean_relevance_score_fn',
 '_get_retriever_tags',
 '_max_inner_product_relevance_score_fn',
 '_select_relevance_score_fn',
 '_similarity_search_with_relevance_scores',
 'aadd_documents',
 'aadd_texts',
 'a

In [ ]:
vectorstore

In [ ]:
vectorstore.similarity_search(query="scientists and dinosausrs")

[Document(id='23b86b4e-308f-46a9-82ca-2532dd0e3cae', metadata={'source': 'https://en.wikipedia.org/wiki/Jurassic_Park'}, page_content='A bunch of scientists bring back dinosaurs and mayhem breaks loose'),
 Document(id='a580aee1-0ea5-4895-b3ea-17945d0e8edc', metadata={'source': 'https://en.wikipedia.org/wiki/Oppenheimer_(film)'}, page_content='Leo DiCaprio gets lost in a dream within a dream within a dream within a ...'),
 Document(id='391f7da7-e671-450f-80ca-f82c7d1371db', metadata={'source': 'https://en.wikipedia.org/wiki/Oppenheimer_(film)'}, page_content='Leo DiCaprio gets lost in a dream within a dream within a dream within a ...')]

In [ ]:
await vectorstore.asimilarity_search(query="scientists and dinosausrs")

[Document(id='23b86b4e-308f-46a9-82ca-2532dd0e3cae', metadata={'source': 'https://en.wikipedia.org/wiki/Jurassic_Park'}, page_content='A bunch of scientists bring back dinosaurs and mayhem breaks loose'),
 Document(id='a580aee1-0ea5-4895-b3ea-17945d0e8edc', metadata={'source': 'https://en.wikipedia.org/wiki/Oppenheimer_(film)'}, page_content='Leo DiCaprio gets lost in a dream within a dream within a dream within a ...'),
 Document(id='391f7da7-e671-450f-80ca-f82c7d1371db', metadata={'source': 'https://en.wikipedia.org/wiki/Oppenheimer_(film)'}, page_content='Leo DiCaprio gets lost in a dream within a dream within a dream within a ...')]

In [ ]:
embedding = embeddings.embed_query("dinosaurs")
vectorstore.similarity_search_by_vector(embedding)

[Document(id='23b86b4e-308f-46a9-82ca-2532dd0e3cae', metadata={'source': 'https://en.wikipedia.org/wiki/Jurassic_Park'}, page_content='A bunch of scientists bring back dinosaurs and mayhem breaks loose'),
 Document(id='a580aee1-0ea5-4895-b3ea-17945d0e8edc', metadata={'source': 'https://en.wikipedia.org/wiki/Oppenheimer_(film)'}, page_content='Leo DiCaprio gets lost in a dream within a dream within a dream within a ...'),
 Document(id='391f7da7-e671-450f-80ca-f82c7d1371db', metadata={'source': 'https://en.wikipedia.org/wiki/Oppenheimer_(film)'}, page_content='Leo DiCaprio gets lost in a dream within a dream within a dream within a ...')]

In [ ]:
from langchain_core.documents import Document
from langchain_core.runnables import RunnableLambda

retriever = RunnableLambda(vectorstore.similarity_search).bind(k=1)
retriever.batch(["scientists and dinosaurs"])

[[Document(id='23b86b4e-308f-46a9-82ca-2532dd0e3cae', metadata={'source': 'https://en.wikipedia.org/wiki/Jurassic_Park'}, page_content='A bunch of scientists bring back dinosaurs and mayhem breaks loose')]]

In [ ]:
!pip install -Uq langchain-google-genai

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", api_key=userdata.get('GOOGLE_API_KEY'))
#

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

message = """
Answer this question using provided context only
(question)

(context)
"""


In [ ]:
prompt = ChatPromptTemplate.from_messages([("human", message)])

RAG

In [ ]:
rag_chain = {"context": retriever, "question": RunnablePassthrough()} | prompt | llm

In [ ]:
response = rag_chain.invoke("Who am I")
print(response.content)

I cannot answer the question because both the question itself and the context are missing. Please provide them.


In [ ]:
response = rag_chain.invoke("tell me about dinosaurs")
print(response.content)

I cannot answer the question because both the `(question)` and `(context)` were left empty. Please provide the question and the context.
